In [1]:


import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter






In [2]:
# ==============================================================================
# --- SECCIÓN 1: FUNCIONES DE AYUDA PARA ANÁLISIS Y VISUALIZACIÓN ---
# ==============================================================================

def analizar_y_visualizar_red(G, titulo_red, layout_func=nx.spring_layout, es_bipartita=False, tipo_nodo_col=None):
    """
    Función genérica para analizar métricas clave y generar una visualización de una red.
    """
    print(f"\n--- Análisis de la Red: {titulo_red} ---")
    
    # 1. Información básica de la red
    num_nodos = G.number_of_nodes()
    num_aristas = G.number_of_edges()
    print(f"Número de Nodos: {num_nodos}")
    print(f"Número de Aristas: {num_aristas}")

    if not G.nodes():
        print("La red está vacía. No se puede continuar el análisis.")
        return

    # Si la red no es bipartita, realizamos análisis más profundos
    if not es_bipartita:
        # 2. Componentes Conexas
        num_componentes = nx.number_connected_components(G)
        print(f"Número de Componentes Conexas: {num_componentes}")
        
        # Tomamos la componente gigante para análisis más detallados
        componentes = sorted(nx.connected_components(G), key=len, reverse=True)
        G_gigante = G.subgraph(componentes[0])
        print(f"Tamaño de la Componente Gigante: {G_gigante.number_of_nodes()} nodos")

        # 3. Detección de Comunidades (usando el algoritmo de Louvain)
        print("\nDetectando comunidades...")
        comunidades = nx.community.louvain_communities(G_gigante, weight='weight' if 'weight' in list(G.edges(data=True))[0][-1] else None)
        print(f"Se encontraron {len(comunidades)} comunidades en la componente gigante.")
        
        # Asignar a cada nodo su comunidad para colorear
        colores_nodos = {}
        for i, comunidad in enumerate(comunidades):
            for nodo in comunidad:
                colores_nodos[nodo] = i

    # 4. Visualización
    print("Generando visualización...")
    plt.figure(figsize=(20, 20))
    pos = layout_func(G) # Calcular posiciones de los nodos

    # Configuración de colores y tamaños
    if es_bipartita and tipo_nodo_col:
        colores = ['skyblue' if G.nodes[n][tipo_nodo_col] == 'Autor' else 'lightgreen' for n in G.nodes()]
        nodos_autores = [n for n, d in G.nodes(data=True) if d[tipo_nodo_col] == 'Autor']
        nodos_otros = [n for n, d in G.nodes(data=True) if d[tipo_nodo_col] != 'Autor']
        node_size_map = {n: 50 for n in nodos_autores}
        node_size_map.update({n: 200 for n in nodos_otros})
        sizes = [node_size_map.get(n, 100) for n in G.nodes()]
    elif not es_bipartita:
        colores = [colores_nodos.get(n, -1) for n in G.nodes()] # Usar colores de comunidad
        grados = dict(G.degree())
        sizes = [grados.get(n, 1) * 50 for n in G.nodes()] # Tamaño por grado
    else:
        colores = 'skyblue'
        sizes = 100

    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colores, cmap=plt.cm.jet, alpha=0.8)
    
    # Dibujar aristas con pesos (si existen)
    if 'weight' in list(G.edges(data=True))[0][-1]:
        pesos = [d['weight'] for (u, v, d) in G.edges(data=True)]
        nx.draw_networkx_edges(G, pos, width=[w * 0.5 for w in pesos], alpha=0.3, edge_color='gray')
    else:
        nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='gray')

    # Etiquetas para nodos más grandes (para no saturar el gráfico)
    nodos_grandes = [n for n, size in zip(G.nodes(), sizes) if size > sorted(sizes)[-20:].pop(0)]
    etiquetas = {n: n for n in nodos_grandes}
    nx.draw_networkx_labels(G, pos, labels=etiquetas, font_size=10, font_color='black')
    
    plt.title(titulo_red, size=20)
    plt.axis('off')
    plt.savefig(f"{titulo_red.replace(' ', '_').lower()}.png", bbox_inches='tight')
    print(f"Gráfico guardado como '{titulo_red.replace(' ', '_').lower()}.png'")
    plt.close()

In [ ]:
# ==============================================================================
# --- SECCIÓN 2: CARGA Y ANÁLISIS DE CADA RED ---
# ==============================================================================

# --- 1. Análisis de Red de Coautoría ---
try:
    df_nodos_coautoria = pd.read_csv('1_red_coautoria_nodos.csv')
    df_aristas_coautoria = pd.read_csv('1_red_coautoria_aristas.csv')
    
    G_coautoria = nx.from_pandas_edgelist(df_aristas_coautoria, 'Source', 'Target', edge_attr='Weight')
    analizar_y_visualizar_red(G_coautoria, "Red de Coautoria")
    
    # Análisis de centralidad adicional
    print("\nTop 10 Autores por Centralidad de Grado:")
    grado = sorted(G_coautoria.degree(weight='Weight'), key=lambda item: item[1], reverse=True)
    print(pd.DataFrame(grado[:10], columns=['Autor', 'Grado Ponderado']))
    
    print("\nTop 10 Autores por Centralidad de Intermediación:")
    intermediacion = nx.betweenness_centrality(G_coautoria, weight='Weight', normalized=True)
    intermediacion_sorted = sorted(intermediacion.items(), key=lambda item: item[1], reverse=True)
    print(pd.DataFrame(intermediacion_sorted[:10], columns=['Autor', 'Intermediación']))
    
except FileNotFoundError:
    print("Archivos para la Red de Coautoría no encontrados. Saltando análisis.")


# --- 2. Análisis de Red de Afiliación (Bipartita) ---
try:
    df_nodos_afiliacion = pd.read_csv('redes/1_red_coautoria_aristas.csv')
    df_aristas_afiliacion = pd.read_csv('redes/2_red_afiliacion_aristas.csv')

    G_afiliacion = nx.from_pandas_edgelist(df_aristas_afiliacion, 'Source', 'Target')
    # Añadir atributos de tipo de nodo
    nx.set_node_attributes(G_afiliacion, pd.Series(df_nodos_afiliacion.TipoNodo, index=df_nodos_afiliacion.Id).to_dict(), 'TipoNodo')
    
    analizar_y_visualizar_red(G_afiliacion, "Red de Afiliacion Autor-Institucion", es_bipartita=True, tipo_nodo_col='TipoNodo', layout_func=nx.kamada_kawai_layout)
    
except FileNotFoundError:
    print("Archivos para la Red de Afiliación no encontrados. Saltando análisis.")


# --- 3. Análisis de Red de Palabras Clave ---
try:
    df_nodos_kw = pd.read_csv('redes/3_red_keywords_nodos.csv')
    df_aristas_kw = pd.read_csv('redes/3_red_keywords_aristas.csv')
    
    G_kw = nx.from_pandas_edgelist(df_aristas_kw, 'Source', 'Target', edge_attr='Weight')
    analizar_y_visualizar_red(G_kw, "Red de Co-ocurrencia de Palabras Clave")
    
    print("\nTop 10 Palabras Clave por Centralidad de Grado:")
    grado_kw = sorted(G_kw.degree(weight='Weight'), key=lambda item: item[1], reverse=True)
    print(pd.DataFrame(grado_kw[:10], columns=['Palabra Clave', 'Grado Ponderado']))
    
except FileNotFoundError:
    print("Archivos para la Red de Palabras Clave no encontrados. Saltando análisis.")


# --- 4. Análisis de Red Temática (Bipartita) ---
try:
    df_nodos_tematica = pd.read_csv('redes/4_red_tematica_nodos.csv')
    df_aristas_tematica = pd.read_csv('redes/4_red_tematica_aristas.csv')
    
    G_tematica = nx.from_pandas_edgelist(df_aristas_tematica, 'Source', 'Target')
    nx.set_node_attributes(G_tematica, pd.Series(df_nodos_tematica.TipoNodo, index=df_nodos_tematica.Id).to_dict(), 'TipoNodo')

    analizar_y_visualizar_red(G_tematica, "Red de Afiliacion Autor-Tematica", es_bipartita=True, tipo_nodo_col='TipoNodo', layout_func=nx.kamada_kawai_layout)

except FileNotFoundError:
    print("Archivos para la Red Temática no encontrados. Saltando análisis.")

print("\nAnálisis completado.")

Archivos para la Red de Coautoría no encontrados. Saltando análisis.
Archivos para la Red de Afiliación no encontrados. Saltando análisis.
Archivos para la Red de Palabras Clave no encontrados. Saltando análisis.
Archivos para la Red Temática no encontrados. Saltando análisis.

Análisis completado.
